In [1]:
# Uncomment if needed
# !pip install transformers torch pandas matplotlib

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

In [2]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(type(tokenizer))
print(type(model))

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vinna\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

<class 'transformers.models.distilbert.tokenization_distilbert_fast.DistilBertTokenizerFast'>
<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>


In [3]:
text = "I absolutely loved this movie. The acting was fantastic!"

print("Input Text:")
print(text)

Input Text:
I absolutely loved this movie. The acting was fantastic!


In [4]:
encoding = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

print(encoding)

{'input_ids': tensor([[  101,  1045,  7078,  3866,  2023,  3185,  1012,  1996,  3772,  2001,
         10392,   999,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [5]:
print("Input IDs:")
print(encoding["input_ids"])

print("\nAttention Mask:")
print(encoding["attention_mask"])

Input IDs:
tensor([[  101,  1045,  7078,  3866,  2023,  3185,  1012,  1996,  3772,  2001,
         10392,   999,   102]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [6]:
tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

print("Tokens:")
print(tokens)

Tokens:
['[CLS]', 'i', 'absolutely', 'loved', 'this', 'movie', '.', 'the', 'acting', 'was', 'fantastic', '!', '[SEP]']


In [7]:
print("========== TOKENIZER OUTPUT ==========")
for token, idx in zip(tokens, encoding["input_ids"][0].tolist()):
    print(f"{token:15} -> {idx}")

========== TOKENIZER OUTPUT ==========
[CLS]           -> 101
i               -> 1045
absolutely      -> 7078
loved           -> 3866
this            -> 2023
movie           -> 3185
.               -> 1012
the             -> 1996
acting          -> 3772
was             -> 2001
fantastic       -> 10392
!               -> 999
[SEP]           -> 102


In [8]:
embedding_layer = model.distilbert.embeddings.word_embeddings

input_ids = encoding["input_ids"]

embeddings = embedding_layer(input_ids)

print("Embedding Tensor Shape:")
print(embeddings.shape)

Embedding Tensor Shape:
torch.Size([1, 13, 768])


In [9]:
print("Embedding vector for first token:")
print(embeddings[0][0][:20])  # first 20 values

Embedding vector for first token:
tensor([ 0.0399, -0.0102, -0.0204,  0.0007, -0.0177,  0.0412, -0.0204,  0.0041,
        -0.0215, -0.0449,  0.0124, -0.0139,  0.0069, -0.0137, -0.0407,  0.0230,
         0.0434,  0.0604, -0.0032, -0.0358], grad_fn=<SliceBackward0>)


In [10]:
print("One token becomes a vector of length:")
print(embeddings.shape[-1])

One token becomes a vector of length:
768


In [11]:
outputs = model(**encoding, output_hidden_states=True)

In [12]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-4.3283,  4.6765]], grad_fn=<AddmmBackward0>), hidden_states=(tensor([[[ 0.3549, -0.1386, -0.2253,  ...,  0.1536,  0.0748,  0.1310],
         [ 0.2282,  0.5511, -0.5092,  ...,  0.6421,  0.9541,  0.3192],
         [ 0.8940, -0.0292, -0.3731,  ...,  0.3402,  0.2767,  0.2639],
         ...,
         [ 0.1993,  0.8991,  0.1944,  ...,  0.6175, -0.1293, -0.2954],
         [ 0.7747, -0.1034, -0.8155,  ...,  0.2032,  0.5654,  0.5736],
         [-0.5912,  0.0365, -0.0826,  ..., -0.1840,  0.0280, -0.0503]]],
       grad_fn=<NativeLayerNormBackward0>), tensor([[[ 0.0373,  0.0824, -0.1061,  ...,  0.0592,  0.0042,  0.0192],
         [ 0.5279,  0.8558, -0.2593,  ...,  0.3113,  0.4716,  0.5997],
         [ 1.7962,  1.0251,  0.2060,  ..., -0.2547,  0.3955,  0.3597],
         ...,
         [ 1.0349,  1.5698,  0.4935,  ..., -0.4155, -0.0737, -0.5038],
         [-0.0503,  0.3828, -0.5784,  ...,  0.3301,  0.0578,  0.8454],
         [-0.2611,  0.0113, -0.

In [13]:
hidden_states = outputs.hidden_states

print("Number of hidden state tensors:")
print(len(hidden_states))

Number of hidden state tensors:
7


In [14]:
last_hidden = hidden_states[-1]

print(last_hidden.shape)

torch.Size([1, 13, 768])


In [15]:
logits = outputs.logits

print("Logits:")
print(logits)

Logits:
tensor([[-4.3283,  4.6765]], grad_fn=<AddmmBackward0>)


In [16]:
probabilities = F.softmax(logits, dim=1)

print("Probabilities:")
print(probabilities)

Probabilities:
tensor([[1.2281e-04, 9.9988e-01]], grad_fn=<SoftmaxBackward0>)


In [17]:
labels = model.config.id2label

for idx, prob in enumerate(probabilities[0]):
    print(f"{labels[idx]:10} : {prob.item():.4f}")

NEGATIVE   : 0.0001
POSITIVE   : 0.9999


In [18]:
predicted_class = torch.argmax(probabilities, dim=1).item()

predicted_label = labels[predicted_class]

print("=" * 50)
print("Final Prediction:")
print(predicted_label)
print("=" * 50)

Final Prediction:
POSITIVE


In [19]:
def explain_prediction(text):
    print("=" * 60)
    print("INPUT:")
    print(text)
    print("=" * 60)

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

    print("\nTOKENS:")
    print(tokens)

    outputs = model(**encoding, output_hidden_states=True)

    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    print("\nLOGITS:")
    print(logits)

    print("\nPROBABILITIES:")
    for idx, prob in enumerate(probs[0]):
        print(f"{model.config.id2label[idx]:10} : {prob.item():.4f}")

    prediction = model.config.id2label[torch.argmax(probs, dim=1).item()]

    print("\nFINAL PREDICTION:")
    print(prediction)

In [20]:
explain_prediction("The movie was absolutely wonderful and inspiring!")

print("\n")

explain_prediction("This was the worst film I have ever watched.")

INPUT:
The movie was absolutely wonderful and inspiring!

TOKENS:
['[CLS]', 'the', 'movie', 'was', 'absolutely', 'wonderful', 'and', 'inspiring', '!', '[SEP]']

LOGITS:
tensor([[-4.3770,  4.7310]], grad_fn=<AddmmBackward0>)

PROBABILITIES:
NEGATIVE   : 0.0001
POSITIVE   : 0.9999

FINAL PREDICTION:
POSITIVE


INPUT:
This was the worst film I have ever watched.

TOKENS:
['[CLS]', 'this', 'was', 'the', 'worst', 'film', 'i', 'have', 'ever', 'watched', '.', '[SEP]']

LOGITS:
tensor([[ 4.6045, -3.6943]], grad_fn=<AddmmBackward0>)

PROBABILITIES:
NEGATIVE   : 0.9998
POSITIVE   : 0.0002

FINAL PREDICTION:
NEGATIVE
